<img src="https://ga-curriculum.github.io/universal-resources-internal/ga-logos/ga-red-cog-72x72.png" alt="" style="float: left; margin: 20px 20px 0 0; height: 55px">

# Data Augmentation

_Author: Tim Book_

In [1]:
import os
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow import keras
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom
from tensorflow.keras.utils import image_dataset_from_directory

In [2]:
train_labels = pd.read_csv('butterflies/Training_set.csv')
C = train_labels.label.nunique()

Create datasets as in the previous section.

In [3]:
train_ds = image_dataset_from_directory(
    'data/train',
    batch_size=32,
    subset='training',
    seed=123,
    validation_split=0.2
).map(lambda x, y: (x/255, y))

val_ds = image_dataset_from_directory(
    'data/train',
    batch_size=32,
    subset='validation',
    seed=123,
    validation_split=0.2
).map(lambda x, y: (x/255, y))

Found 6499 files belonging to 75 classes.
Using 5200 files for training.
Found 6499 files belonging to 75 classes.
Using 1299 files for validation.


In [4]:
model = keras.models.Sequential([
    RandomFlip('horizontal_and_vertical', input_shape=(256, 256, 3)),
    RandomRotation(0.2),
    RandomZoom(height_factor=(-0.2, 0.2), width_factor=(-0.2, 0.2)),
    Conv2D(32, 3, activation='relu'),
    MaxPooling2D(2),
    Conv2D(32, 3, activation='relu'),
    MaxPooling2D(2),
    Flatten(),
    Dense(16, activation='relu'),
    Dropout(0.5),
    Dense(C, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='rmsprop',
    metrics=['acc']
)

In [5]:
model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
random_flip (RandomFlip)     (None, 256, 256, 3)       0         
_________________________________________________________________
random_rotation (RandomRotat (None, 256, 256, 3)       0         
_________________________________________________________________
random_zoom (RandomZoom)     (None, 256, 256, 3)       0         
_________________________________________________________________
conv2d (Conv2D)              (None, 254, 254, 32)      896       
_________________________________________________________________
max_pooling2d (MaxPooling2D) (None, 127, 127, 32)      0         
_________________________________________________________________
conv2d_1 (Conv2D)            (None, 125, 125, 32)      9248      
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 62, 62, 32)        0

In [6]:
model.fit(train_ds, validation_data=val_ds, epochs=5)

Epoch 1/5
 67/163 [===========>..................] - ETA: 1:10 - loss: 4.4923 - acc: 0.0163

KeyboardInterrupt: 